# Pillar 1 - Phase 1 baseline v2 (bigger samples)

See `plans/PLAN.md` (Tru cot 1) and `phases/phase-1-no-train.md`. Same setup as v1
(`runs/2026_09_12_pillar1_phase1_kaggle_baseline/`) but with larger word lists / more
text samples for more reliable ratios, plus a top_k_frac sensitivity check for 1.2.

Runs on Kaggle CPU (sequences are short, no GPU needed).


In [ ]:
import torch, json, statistics, re
from transformers import AutoTokenizer, AutoModelForCausalLM
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# 1.1: word lists (baseline = common disyllabic words)
common_words = [
    "con người", "học sinh", "gia đình", "công việc", "thời gian", "buổi sáng",
    "cuộc sống", "quê hương", "bạn bè", "trường học", "thành phố", "đất nước",
    "sức khỏe", "niềm vui", "câu chuyện", "con đường", "bữa cơm", "giấc ngủ",
    "điện thoại", "máy tính", "xe máy", "bệnh viện", "công viên", "siêu thị",
    "ngân hàng", "sân bay", "nhà hàng", "khách sạn", "bãi biển", "núi rừng",
    "dòng sông", "cánh đồng", "mùa xuân", "mùa hè", "mùa thu", "mùa đông",
    "buổi tối", "buổi trưa", "buổi chiều", "hôm nay", "ngày mai", "tuần sau",
    "tháng trước", "năm nay",
]

rare_compounds = [
    "khấp khiễng", "khệnh khạng", "lẩn khuất", "chông chênh", "khúm núm",
    "ngúng nguẩy", "rón rén", "khù khờ", "lù đù", "ngơ ngác", "chênh vênh",
    "lấp lửng", "ngổn ngang", "khắc khoải", "dùng dằng", "thất thểu",
    "ngật ngưỡng", "chán chường", "khấp khởi", "lởn vởn", "khệ nệ",
    "lừng khừng", "ngơ ngẩn", "chán ngắt", "khờ khạo", "lóng ngóng",
    "hấp tấp", "lụp xụp", "xộc xệch", "tuềnh toàng", "lem nhem",
    "nhem nhuốc", "xoàng xĩnh", "lạch bạch", "lù rù", "rề rà",
    "hối hả", "tất bật", "bộn bề", "chật vật", "lận đận",
    "lao đao", "chếnh choáng", "ngà ngà", "lâng lâng", "bàng hoàng",
    "thẫn thờ", "ngẩn ngơ", "thơ thẩn", "vơ vẩn", "lan man",
    "miên man", "mải miết", "miệt mài", "hì hục", "lúi húi",
    "cặm cụi", "hùng hục",
]

proper_nouns = [
    "Ea H'leo", "Mường Lát", "Pác Nặm", "Bát Xát", "Na Hang", "Trùng Khánh",
    "Xín Mần", "Mèo Vạc", "Đắk Glei", "Krông Pắk", "Nậm Pồ", "Mường Nhé",
    "Sìn Hồ", "Mường Tè", "Tủa Chùa", "Bảo Lạc", "Hà Quảng", "Quản Bạ",
    "Yên Minh", "Đồng Văn", "Hoàng Su Phì", "Pú Nhung", "Chiềng Yên",
    "Bắc Yên", "Phù Yên", "Trạm Tấu", "Mù Cang Chải", "A Lưới", "Nam Đông",
    "Tây Trà", "Kon Plông", "Đắk Tô", "Ea Súp", "Cư M'gar", "Buôn Đôn",
    "Krông Ana", "Ea Kar", "Ia Grai", "Chư Prông", "Đức Cơ",
    "Nguyễn Thị Thanh Vân", "Đinh Xuân Bá", "Vừ A Sính", "Hoàng Thị Kim Chi",
    "Lò Văn Sang", "Giàng Seo Phử", "Sùng A Của", "Hờ A Súa",
    "Vàng Thị Máy", "Thào A Tủa", "Mã A Lềnh", "Lò Ngân Sủn",
]

numbers = [
    "12/09/2026", "1.234.567 VNĐ", "$99.99", "MST: 0312345678-001",
    "0987654321", "23:59:59 ngày 31/12/2025", "Điều 12, Khoản 3",
    "3.14159265", "100.000.000 đồng", "GD-2026-00123", "+84 912 345 678",
    "50%/năm", "km số 27+500", "1.5 triệu", "2026-09-12", "12h30",
    "0938.123.456", "STK: 19036789012345", "CMND: 079099001234",
    "Lô A1-05", "1/3", "3/4 khối lượng", "±5%", "10^-6", "2^10 = 1024",
    "Mã vùng: 028", "IPv4: 192.168.1.1", "Lãi suất 8.5%/năm",
    "Diện tích 54.2 m²", "Nhiệt độ -3°C",
]

typo_pairs = [
    (
        "Hôm nay trời đẹp, chúng tôi cùng nhau đi dạo quanh hồ và trò chuyện về "
        "những dự định trong tương lai. Ai cũng cảm thấy vui vẻ và tràn đầy hy vọng.",
        "Hom nay troi dep, chung toi cung nhau di dao quanh ho va tro chuyen ve "
        "nhung du dinh trong tuong lai. Ai cung cam thay vui ve va tran day hy vong.",
    ),
    (
        "Sáng nay giá vàng trong nước tăng mạnh sau khi thị trường thế giới biến động, "
        "khiến nhiều nhà đầu tư đổ xô đi mua vào bất chấp rủi ro.",
        "Sang nay gia vang trong nuoc tang manh sau khi thi truong the gioi bien dong, "
        "khien nhieu nha dau tu do xo di mua vao bat chap rui ro.",
    ),
    (
        "Cho hai quả trứng vào bát, đánh tan cùng chút muối tiêu, sau đó đổ vào chảo "
        "dầu nóng và chiên đều hai mặt cho đến khi vàng ươm.",
        "Cho hai qua trung vao bat, danh tan cung chut muoi tieu, sau do do vao chao "
        "dau nong va chien deu hai mat cho den khi vang uom.",
    ),
    (
        "Kính gửi anh chị, em xin gửi lại bản báo cáo đã chỉnh sửa theo góp ý, rất "
        "mong nhận được phản hồi sớm để kịp tiến độ dự án.",
        "Kinh gui anh chi, em xin gui lai ban bao cao da chinh sua theo gop y, rat "
        "mong nhan duoc phan hoi som de kip tien do du an.",
    ),
    (
        "Chuyến đi lên vùng cao lần này để lại cho tôi nhiều cảm xúc khó quên, từ "
        "con đường quanh co đến những thửa ruộng bậc thang trải dài.",
        "Chuyen di len vung cao lan nay de lai cho toi nhieu cam xuc kho quen, tu "
        "con duong quanh co den nhung thua ruong bac thang trai dai.",
    ),
]


In [ ]:
# NlpHUST/gpt2-vietnamese instead of vinai/PhoGPT-4B: PhoGPT's MPT config raises
# StrictDataclassFieldValidationError on current transformers (attn_pdrop int vs float).
bpe_tok = AutoTokenizer.from_pretrained("NlpHUST/gpt2-vietnamese")

def syllables(s):
    return len(s.split())

def tokens(s, tok):
    return len(tok.encode(s, add_special_tokens=False))

def ratio_stats(words, tok):
    ratios = [tokens(w, tok) / max(syllables(w), 1) for w in words]
    return {"mean": statistics.mean(ratios), "stdev": statistics.stdev(ratios), "n": len(ratios)}

baseline = ratio_stats(common_words, bpe_tok)
results_1_1 = {"baseline": baseline}
for name, wl in [("1_1a_rare_compounds", rare_compounds),
                 ("1_1b_proper_nouns", proper_nouns),
                 ("1_1c_numbers", numbers)]:
    r = ratio_stats(wl, bpe_tok)
    results_1_1[name] = {
        **r,
        "multiplier_vs_baseline": r["mean"] / baseline["mean"],
        "pass_threshold_2x": (r["mean"] / baseline["mean"]) > 2.0,
    }

typo_ratios = [tokens(noisy, bpe_tok) / tokens(clean, bpe_tok) for clean, noisy in typo_pairs]
results_1_1["1_1d_typos"] = {
    "mean_multiplier": statistics.mean(typo_ratios),
    "stdev": statistics.stdev(typo_ratios),
    "n_pairs": len(typo_ratios),
    "pass_threshold_2x": statistics.mean(typo_ratios) > 2.0,
}

for k, v in results_1_1.items():
    print(k, "->", v)


In [ ]:
# 1.2: entropy peak vs syllable boundary, averaged over multiple texts + 2 top_k_frac values
sample_texts = [
    "Việt Nam là một quốc gia nằm ở khu vực Đông Nam Á, có bờ biển dài và "
    "nền văn hóa lâu đời. Người dân nơi đây rất cần cù và hiếu khách.",
    "Sáng sớm, mặt trời vừa ló rạng sau rặng núi, ánh nắng chiếu xuống thung "
    "lũng khiến sương mù dần tan biến, để lộ những thửa ruộng bậc thang xanh mướt.",
    "Anh có thể chỉ giúp tôi đường đến bưu điện gần nhất không? Tôi cần gửi "
    "một bưu kiện trước năm giờ chiều nay, cảm ơn anh rất nhiều.",
    "Trước tiên, hãy rửa sạch rau củ rồi thái nhỏ vừa ăn, sau đó ướp gia vị "
    "trong khoảng mười lăm phút trước khi cho lên bếp xào ở lửa lớn.",
    "Ông lão ngồi lặng lẽ bên hiên nhà, nhìn đàn trẻ nô đùa ngoài sân mà lòng "
    "bồi hồi nhớ về những ngày xưa cũ, khi làng xóm còn chưa đổi thay nhiều.",
]

def syllable_boundary_positions(text):
    positions = {m.end() for m in re.finditer(r"\s", text)}
    positions.add(0)
    return positions

@torch.no_grad()
def entropy_peak_overlap(model_name, text, top_k_frac):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
    model.eval()

    enc = tok(text, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    logits = model(**enc).logits[0]
    probs = torch.softmax(logits.float(), dim=-1)
    entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum(-1)
    entropy = entropy.cpu().tolist()

    char_positions = [end for (_, end) in offsets]
    gt = syllable_boundary_positions(text)
    n_peaks = max(int(len(entropy) * top_k_frac), 1)
    peak_idx = sorted(range(len(entropy)), key=lambda i: entropy[i], reverse=True)[:n_peaks]
    peak_char_positions = {char_positions[i] for i in peak_idx}

    hit = sum(1 for p in peak_char_positions if any(abs(p - b) <= 1 for b in gt))
    return hit / max(len(peak_char_positions), 1) * 100

def eval_model_over_texts(model_name, texts, top_k_frac):
    overlaps = [entropy_peak_overlap(model_name, t, top_k_frac) for t in texts]
    return {"mean": statistics.mean(overlaps), "stdev": statistics.stdev(overlaps),
            "n_texts": len(overlaps), "per_text": overlaps}

results_1_2 = {}
for frac in (0.2, 0.3):
    results_1_2[f"1_2a_xglm564M_top{int(frac*100)}pct"] = eval_model_over_texts(
        "facebook/xglm-564M", sample_texts, frac)
    results_1_2[f"1_2b_gpt2_vietnamese_top{int(frac*100)}pct"] = eval_model_over_texts(
        "NlpHUST/gpt2-vietnamese", sample_texts, frac)

for k, v in results_1_2.items():
    print(k, "-> mean_overlap_pct:", round(v["mean"], 1), "stdev:", round(v["stdev"], 1))


In [ ]:
import os
out_dir = "/kaggle/working"
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "metrics_pillar1_phase1_v2.jsonl"), "w") as f:
    f.write(json.dumps({"section": "1.1", "results": results_1_1}) + "\n")
    f.write(json.dumps({"section": "1.2", "results": results_1_2}) + "\n")
print("wrote", os.path.join(out_dir, "metrics_pillar1_phase1_v2.jsonl"))
